<a href="https://colab.research.google.com/github/AhmedE16/flyrank-ai-intern-ahmed-essam/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)

import duckdb, os
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get("HF_TOKEN")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
REL = "hf://datasets/FlyRank/internship-warehouse"

# Page-level month=2026-03 aggregate (same grain as w03)
page_month = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions_month,
        SUM(gsc_clicks) AS clicks_month,
        AVG(gsc_avg_position) AS avg_position
    FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
    WHERE month = '2026-03'
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

page_month["ctr"] = page_month["clicks_month"] / page_month["impressions_month"]
page_month["position_tier"] = pd_cut = None
import pandas as pd
page_month["position_tier"] = pd.cut(
    page_month["avg_position"], bins=[0, 3, 10, 20, 1000],
    labels=["1-3", "4-10", "11-20", "21+"]
)

# SIGNAL 1 -- CTR vs position tier (behind the CTR-fix flag logic)
signal1 = page_month.groupby("position_tier", observed=True).agg(
    n=("content_hash_id", "count"),
    avg_ctr=("ctr", "mean")
).round(4)
print("SIGNAL 1: CTR by position tier")
print(signal1)
print("\nVerdict: CONFIRMED -- CTR drops as position tier gets worse, matching the")
print("assumption behind FlyRank's low_ctr_visible_page / CTR-fix flag.\n")

# SIGNAL 2 -- volume vs click-through (behind the quick-win / volume assumption)
page_month["volume_tier"] = pd.qcut(page_month["impressions_month"], 4, labels=["low", "mid", "high", "very_high"])
signal2 = page_month.groupby("volume_tier", observed=True).agg(
    n=("content_hash_id", "count"),
    avg_ctr=("ctr", "mean")
).round(4)
print("SIGNAL 2: CTR by impression-volume tier")
print(signal2)
print("\nVerdict: MIXED -- CTR does not rise cleanly with volume; high-impression pages")
print("are not automatically better at converting impressions to clicks. Raw volume")
print("alone is not a safe stand-in for 'this page is doing fine', which is exactly")
print("why a quick-win flag based on volume alone would be too blunt on its own.\n")

print("""
MY RULE (plain words):
Flag a page as a "CTR review candidate" if it gets meaningful search visibility
(impressions_month >= 500) AND sits in a position where it should be getting more
clicks (avg_position between 1 and 20) AND its actual CTR falls below what Signal 1
showed is typical for its own position tier. This targets Signal 1's CONFIRMED gap
directly, and avoids Signal 2's MIXED volume assumption by not using raw volume as
the deciding factor -- only as a visibility filter.

Reason code: low_ctr_visible_page
Action: review_title_meta
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SIGNAL 1: CTR by position tier
                   n  avg_ctr
position_tier                
1-3            16144   0.0106
4-10           81988   0.0049
11-20          32203   0.0032
21+            44969   0.0019

Verdict: CONFIRMED -- CTR drops as position tier gets worse, matching the
assumption behind FlyRank's low_ctr_visible_page / CTR-fix flag.

SIGNAL 2: CTR by impression-volume tier
                 n  avg_ctr
volume_tier                
low          44983   0.0101
mid          43409   0.0028
high         44186   0.0024
very_high    44160   0.0030

Verdict: MIXED -- CTR does not rise cleanly with volume; high-impression pages
are not automatically better at converting impressions to clicks. Raw volume
alone is not a safe stand-in for 'this page is doing fine', which is exactly
why a quick-win flag based on volume alone would be too blunt on its own.


MY RULE (plain words):
Flag a page as a "CTR review candidate" if it gets meaningful search visibility
(impressions_month >= 500) 

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
# Expected CTR per position tier, from Signal 1 above -- used as the "should be getting" baseline
tier_expected_ctr = signal1["avg_ctr"].to_dict()
page_month["expected_ctr"] = page_month["position_tier"].map(tier_expected_ctr).astype(float)
page_month["ctr_gap"] = page_month["expected_ctr"] - page_month["ctr"]

# The rule, encoded
is_candidate = (
    (page_month["impressions_month"] >= 500) &
    (page_month["avg_position"] > 0) & (page_month["avg_position"] <= 20) &
    (page_month["ctr"] < page_month["expected_ctr"])
)

queue = page_month[is_candidate].copy()
queue["reason_code"] = "low_ctr_visible_page"
queue["action"] = "review_title_meta"
queue["score"] = queue["ctr_gap"] * queue["impressions_month"]  # bigger gap + more visibility = higher priority

queue = queue.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Ranked queue built: {len(queue):,} candidate pages out of {len(page_month):,} total.")
print(f"Written to work/outputs/baseline_action_score.csv\n")
print(queue[["content_hash_id", "impressions_month", "avg_position", "ctr", "expected_ctr", "score"]].head(10))

Ranked queue built: 40,823 candidate pages out of 176,738 total.
Written to work/outputs/baseline_action_score.csv

            content_hash_id  impressions_month  avg_position       ctr  \
0  content_8d7d99f109e19aa2           203497.0      2.563756  0.001420   
1  content_0e03de7680314cd5           221310.0      2.675217  0.003253   
2  content_4ffe18112a5642e3           186983.0      2.331060  0.003134   
3  content_ec2e0346994fb5a5           245276.0      2.854514  0.006034   
4  content_44f34c0a90047651           212404.0      7.346909  0.000113   
5  content_545bb6cc7081ded3           122905.0      2.615390  0.002335   
6  content_eadb33b5df496f4a           617124.0      2.383011  0.009185   
7  content_9ef3d7516483e665            89229.0      2.481596  0.001031   
8  content_306bc78dff1eb683            80821.0      1.488604  0.000433   
9  content_c46df0fa61530d86            70398.0      1.556258  0.000597   

   expected_ctr      score  
0        0.0106  1868.0682  
1        0.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
top10 = queue.head(10)

print("TOP-10 REVIEW\n")
for i, row in top10.iterrows():
    print(f"{i+1}. {row['content_hash_id']}")
    print(f"   Action: review_title_meta (reason: low_ctr_visible_page)")
    print(f"   Why: {row['impressions_month']:.0f} impressions at avg position "
          f"{row['avg_position']:.1f}, but CTR {row['ctr']:.3f} vs expected "
          f"{row['expected_ctr']:.3f} for that tier -- a {row['ctr_gap']:.3f} gap.")
    print(f"   What would make this wrong: if this page's low CTR is actually due to "
          f"branded/navigational intent (users already know the destination and skip "
          f"the snippet), not a genuine title/meta problem -- reviewing the actual "
          f"query mix would confirm or rule this out.\n")


TOP-10 REVIEW

1. content_8d7d99f109e19aa2
   Action: review_title_meta (reason: low_ctr_visible_page)
   Why: 203497 impressions at avg position 2.6, but CTR 0.001 vs expected 0.011 for that tier -- a 0.009 gap.
   What would make this wrong: if this page's low CTR is actually due to branded/navigational intent (users already know the destination and skip the snippet), not a genuine title/meta problem -- reviewing the actual query mix would confirm or rule this out.

2. content_0e03de7680314cd5
   Action: review_title_meta (reason: low_ctr_visible_page)
   Why: 221310 impressions at avg position 2.7, but CTR 0.003 vs expected 0.011 for that tier -- a 0.007 gap.
   What would make this wrong: if this page's low CTR is actually due to branded/navigational intent (users already know the destination and skip the snippet), not a genuine title/meta problem -- reviewing the actual query mix would confirm or rule this out.

3. content_4ffe18112a5642e3
   Action: review_title_meta (reason: low

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
# Weak picks -- borderline cases worth a second look (lowest score = weakest justification)
borderline = queue.sort_values("score", ascending=True).head(5)
print("WEAK PICKS -- borderline gap, worth a second look:")
print(borderline[["content_hash_id", "impressions_month", "avg_position", "ctr", "expected_ctr", "score"]])
print("""
These pages barely clear the rule's thresholds -- a low score means either a small
CTR gap or lower visibility, so the case for reviewing them is weak. I'd deprioritize
these relative to the top of the queue rather than treat them with equal confidence.
""")

# Leakage check
print("LEAKAGE CHECK:")
print("- No product flags used (health_score, priority_score, action_type) -- not in this data.")
print("- No future-window data used -- everything comes from month=2026-03 only, the same")
print("  window the score is applied to; no later month's outcome was used to build the rule.")
print("- expected_ctr was computed FROM this same month's data (Signal 1), not from a")
print("  held-out period -- acceptable for a baseline rule, but noting it here so it's not")
print("  mistaken for a validated, leakage-free forward-looking model. That check comes")
print("  in the modeling weeks.")

WEAK PICKS -- borderline gap, worth a second look:
                content_hash_id  impressions_month  avg_position       ctr  \
40822  content_f493fe26c8826691              938.0     12.488547  0.003198   
40821  content_1a74de26965deba9             1429.0      3.249542  0.004899   
40820  content_7be993d834ce79f4             1021.0      3.911260  0.004897   
40819  content_01c6c6de3e5627a2              613.0      3.927326  0.004894   
40817  content_70d30a5863edc3e1              939.0     18.734879  0.003195   

       expected_ctr   score  
40822        0.0032  0.0016  
40821        0.0049  0.0021  
40820        0.0049  0.0029  
40819        0.0049  0.0037  
40817        0.0032  0.0048  

These pages barely clear the rule's thresholds -- a low score means either a small
CTR gap or lower visibility, so the case for reviewing them is weak. I'd deprioritize
these relative to the top of the queue rather than treat them with equal confidence.

LEAKAGE CHECK:
- No product flags used (heal

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.